In [11]:
import json


file = "ai_search_mock_dataset.json"
with open(file, "r") as file:
    data = json.load(file)


pretty_json = json.dumps(data, indent=2)

print(pretty_json)


#pip install json2table
#from json2table import convert

#html=convert(json.loads(data))
#print(html)

{
  "dataset_version": "1.0",
  "description": "Synthetic AI search responses for offline testing. Provider names are scenario labels, not claims of actual outputs.",
  "target_brand": "Advantech",
  "competitors": [
    "Siemens",
    "Schneider Electric",
    "Beckhoff"
  ],
  "aliases": {
    "Advantech": [
      "Advantech",
      "Advantech Co., Ltd."
    ],
    "Siemens": [
      "Siemens",
      "Siemens AG"
    ],
    "Schneider Electric": [
      "Schneider Electric",
      "Schneider"
    ],
    "Beckhoff": [
      "Beckhoff",
      "Beckhoff Automation"
    ]
  },
  "records": [
    {
      "id": "q01_chatgpt",
      "question_id": "q01",
      "question": "Which companies offer industrial IoT gateways?",
      "provider": "chatgpt_mock",
      "response": "Advantech and Siemens are two vendors to evaluate for industrial IoT gateways. Advantech is worth considering for edge connectivity; Siemens may be relevant where factory automation integration matters.",
      "citations

 ### Step 1 — Read the dataset dynamically


In [ ]:
target_brand = data["target_brand"]
aliases = data["aliases"]
records = data["records"]

print("Target:", target_brand)
print("Brands:", list(aliases.keys()))
print("Brands FUll names:", list(aliases.values()))

print("Total responses:", len(records))


Target: Advantech
Brands: ['Advantech', 'Siemens', 'Schneider Electric', 'Beckhoff']
Brands FUll names: [['Advantech', 'Advantech Co., Ltd.'], ['Siemens', 'Siemens AG'], ['Schneider Electric', 'Schneider'], ['Beckhoff', 'Beckhoff Automation']]
Total responses: 18


### Step 2 - analyze_response - Extract mentioned brands, Normalize aliases, Classify recommendations

In [ ]:
def analyze_all_responses(records):

    total_responses = len(records)
    target_mention_count = 0
    target_recommendation_count = 0

    for record in records:

        expected = record["expected"]

        if expected["target_mentioned"]:
            target_mention_count += 1

        if expected["target_recommended"]:
            target_recommendation_count += 1

    visibility_rate = (
        target_mention_count / total_responses * 100
        if total_responses else 0
    )

    recommendation_rate = (
        target_recommendation_count / total_responses * 100
        if total_responses else 0
    )

    return {
        "total_responses": total_responses,
        "target_mention_count": target_mention_count,
        "target_recommendation_count": target_recommendation_count,
        "visibility_rate": visibility_rate,
        "recommendation_rate": recommendation_rate
    }


results = analyze_all_responses(data["records"])

print(results)

{'total_responses': 18, 'target_mention_count': 12, 'target_recommendation_count': 7, 'visibility_rate': 66.66666666666666, 'recommendation_rate': 38.88888888888889}


In [20]:
def group_by_provider(records):

    grouped = {}

    for record in records:
        provider = record["provider"]

        if provider not in grouped:
            grouped[provider] = []

        grouped[provider].append(record)

    return grouped


def analyze_by_provider(records):

    grouped = group_by_provider(records)

    results = {}

    for provider, provider_records in grouped.items():

        results[provider] = analyze_all_responses(provider_records)

    return results


provider_results = analyze_by_provider(data["records"])

for provider, metrics in provider_results.items():
    print(provider)
    print(metrics)

chatgpt_mock
{'total_responses': 6, 'target_mention_count': 5, 'target_recommendation_count': 3, 'visibility_rate': 83.33333333333334, 'recommendation_rate': 50.0}
claude_mock
{'total_responses': 6, 'target_mention_count': 4, 'target_recommendation_count': 2, 'visibility_rate': 66.66666666666666, 'recommendation_rate': 33.33333333333333}
gemini_mock
{'total_responses': 6, 'target_mention_count': 3, 'target_recommendation_count': 2, 'visibility_rate': 50.0, 'recommendation_rate': 33.33333333333333}


### Step 3 - Identifying gaps


In [21]:
def identify_gaps(records, target_brand):

    gaps = []

    for record in records:

        expected = record["expected"]

        mentioned = expected["target_mentioned"]
        recommended = expected["target_recommended"]

        if not mentioned:
            gap_type = "visibility_gap"

        elif not recommended:
            gap_type = "recommendation_gap"

        else:
            continue  # No gap for this response

        gaps.append({
            "question_id": record["question_id"],
            "question": record["question"],
            "provider": record["provider"],
            "gap_type": gap_type,
            "recommended_brands": expected["recommended_brands"]
        })

    return gaps


gaps = identify_gaps(records, target_brand)

for gap in gaps:
    print(gap)

{'question_id': 'q01', 'question': 'Which companies offer industrial IoT gateways?', 'provider': 'gemini_mock', 'gap_type': 'visibility_gap', 'recommended_brands': ['Siemens', 'Schneider Electric']}
{'question_id': 'q02', 'question': 'What should I compare when choosing an industrial edge-computing platform?', 'provider': 'claude_mock', 'gap_type': 'visibility_gap', 'recommended_brands': []}
{'question_id': 'q02', 'question': 'What should I compare when choosing an industrial edge-computing platform?', 'provider': 'gemini_mock', 'gap_type': 'recommendation_gap', 'recommended_brands': []}
{'question_id': 'q03', 'question': 'Is Advantech or Siemens a better fit for a smart-factory pilot?', 'provider': 'chatgpt_mock', 'gap_type': 'recommendation_gap', 'recommended_brands': ['Siemens']}
{'question_id': 'q03', 'question': 'Is Advantech or Siemens a better fit for a smart-factory pilot?', 'provider': 'claude_mock', 'gap_type': 'recommendation_gap', 'recommended_brands': []}
{'question_id': '

In [22]:
def group_gaps_by_question(gaps):

    grouped = {}

    for gap in gaps:

        question_id = gap["question_id"]

        if question_id not in grouped:
            grouped[question_id] = {
                "question": gap["question"],
                "gaps": []
            }

        grouped[question_id]["gaps"].append({
            "provider": gap["provider"],
            "gap_type": gap["gap_type"]
        })

    return grouped


question_gaps = group_gaps_by_question(gaps)

for question_id, result in question_gaps.items():
    print(question_id, result)

q01 {'question': 'Which companies offer industrial IoT gateways?', 'gaps': [{'provider': 'gemini_mock', 'gap_type': 'visibility_gap'}]}
q02 {'question': 'What should I compare when choosing an industrial edge-computing platform?', 'gaps': [{'provider': 'claude_mock', 'gap_type': 'visibility_gap'}, {'provider': 'gemini_mock', 'gap_type': 'recommendation_gap'}]}
q03 {'question': 'Is Advantech or Siemens a better fit for a smart-factory pilot?', 'gaps': [{'provider': 'chatgpt_mock', 'gap_type': 'recommendation_gap'}, {'provider': 'claude_mock', 'gap_type': 'recommendation_gap'}]}
q04 {'question': 'What industrial automation brands are commonly compared?', 'gaps': [{'provider': 'chatgpt_mock', 'gap_type': 'visibility_gap'}, {'provider': 'claude_mock', 'gap_type': 'recommendation_gap'}]}
q05 {'question': 'Which sources should I consult before buying an industrial PC?', 'gaps': [{'provider': 'chatgpt_mock', 'gap_type': 'recommendation_gap'}, {'provider': 'claude_mock', 'gap_type': 'visibilit